# Dependencies

In [1]:
!pip uninstall -y langchain langchain-community chromadb sentence-transformers
!pip install -q \
  langchain==0.1.20 \
  langchain-community==0.0.38 \
  langchain-text-splitters==0.0.1 \
  chromadb==0.4.24 \
  sentence-transformers==2.6.1 \
  pandas==2.2.2


Found existing installation: langchain 0.1.20
Uninstalling langchain-0.1.20:
  Successfully uninstalled langchain-0.1.20
Found existing installation: langchain-community 0.0.38
Uninstalling langchain-community-0.0.38:
  Successfully uninstalled langchain-community-0.0.38
Found existing installation: chromadb 0.4.24
Uninstalling chromadb-0.4.24:
  Successfully uninstalled chromadb-0.4.24
Found existing installation: sentence-transformers 2.6.1
Uninstalling sentence-transformers-2.6.1:
  Successfully uninstalled sentence-transformers-2.6.1



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install langchain-openai

In [ ]:
import os
import pandas as pd

from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
#from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from sentence_transformers import CrossEncoder


# Data path

In [ ]:
DATA_PATH = "clean_customer_support_dataset.csv"

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["Customer_text"]).reset_index(drop=True)

print("Rows:", len(df))


# Making every row a document and then chunking it


In [ ]:
from langchain.schema import Document

def row_to_document(row, idx):
    content = f"""
Issue:
{row.get('Customer_text', '')}

Response:
{row.get('Agent_response', '')}

Category:
{row.get('category', '')}
""".strip()

    return Document(
        page_content=content,
        metadata={
            "row_id": idx,
            "category": row.get("category"),
            "intent": row.get("intent"),
            "source": "clean_customer_support_dataset.csv"
        }
    )

documents = [
    row_to_document(row, idx)
    for idx, row in df.iterrows()
]

print("Documents created:", len(documents))


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=75,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)


In [ ]:
for i, doc in enumerate(documents[:10]):
    print(f"========== DOCUMENT {i} ==========")

    # 1. This is what the AI 'reads' and searches
    print("SEARCHABLE CONTENT (page_content):")
    print(doc.page_content)

    print("\nHIDDEN DATA (metadata):")
    #This is the extra info attached to the chunk
    print(doc.metadata)

    print("\n" + "="*35 + "\n")

In [ ]:
row_0_chunks = [c for c in chunks if c.metadata['row_id'] == 2]

print(f"Original Row 0 was split into {len(row_0_chunks)} chunks.\n")

for i, chunk in enumerate(row_0_chunks):
    print(f"--- CHUNK {i} ---")
    print(chunk.page_content)
    print("-" * 30)

# Embeddings

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

KV_URI = "https://kv-llm-secrets-001.vault.azure.net/"
credential = DefaultAzureCredential()
client = SecretClient(vault_url=KV_URI, credential=credential)

OPENROUTER_API_KEY = client.get_secret("openrouter-api-key").value
APPINSIGHTS_CONNECTION_STRING = client.get_secret("appinsights-connection-string").value

os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
os.environ["OPENAI_ORG_ID"] = "openrouter"


In [ ]:
embedding = OpenAIEmbeddings(
    model="text-embedding-3-large"
)


# Chroma

In [ ]:
CHROMA_DIR = "chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="chroma_db",
    collection_metadata={
        "hnsw:space": "cosine",
        "hnsw:M": 16,
        "hnsw:ef_construction": 100
    }

)


vectorstore.persist()

print("Chroma DB saved to:", CHROMA_DIR)


In [ ]:
!pip install rank_bm25

In [ ]:
dense_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 13}
)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 20

query = "customer refund policy"

dense_results = dense_retriever.invoke(query)
sparse_results = bm25_retriever.invoke(query)


seen = set()
results = []

for d in dense_results + sparse_results:
    if d.page_content not in seen:
        seen.add(d.page_content)
        results.append(d)

for r in results:
    print(r.metadata)
    print(r.page_content[:200])
    print("-" * 40)



In [ ]:
rm -rf lazaboon_chroma_db
python create_lazaboon_db.py

In [ ]:
import shutil

ZIP_NAME = "chroma_db_3nd"
shutil.make_archive(ZIP_NAME, "zip", "chroma_db")
